In [2]:
# Imports

import numpy as np
import pandas as pd
import warnings

from sklearn.metrics import accuracy_score

from art.utils import check_and_transform_label_format
from art.estimators.classification import SklearnClassifier


from art.attacks.evasion import ZooAttack

# Optional: silence the StandardScaler feature-name warning
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but StandardScaler was fitted with feature names",
    category=UserWarning,
)

# IMPORTANT: Import your run_* helpers here.
# Adjust the import paths and function names to match your project.
#
# Example (update to your actual module paths):
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost


In [3]:
# Set seed for reproducibility

SEED = 42
np.random.seed(SEED)

In [4]:
# Run all models on the same dataset path.
# Each run_* function should return (model, X_test, y_test).

DATA_PATH = "CSVs/newDataset.csv"

model_runs = {}

print("Running Logistic Regression...")
logreg_model, logreg_X_test, logreg_y_test = run_logreg(DATA_PATH)
model_runs["LogisticRegression"] = (logreg_model, logreg_X_test, logreg_y_test)

print("Running Neural Net...")
nn_model, nn_X_test, nn_y_test = run_neuralnet(DATA_PATH)
model_runs["NeuralNet"] = (nn_model, nn_X_test, nn_y_test)

print("Running Random Forest...")
rf_model, rf_X_test, rf_y_test = run_randomforest(DATA_PATH)
model_runs["RandomForest"] = (rf_model, rf_X_test, rf_y_test)

print("Running SVM...")
svm_model, svm_X_test, svm_y_test = run_svm(DATA_PATH)
model_runs["SVM"] = (svm_model, svm_X_test, svm_y_test)

print("Running XGBoost...")
xgb_model, xgb_X_test, xgb_y_test = run_xgboost(DATA_PATH)
model_runs["XGBoost"] = (xgb_model, xgb_X_test, xgb_y_test)

print("\nSummary of collected models:")
for name, (model, X_test, y_test) in model_runs.items():
    print(f" - {name}: model={type(model)}, X_test shape={getattr(X_test, 'shape', None)}")

Running Logistic Regression...
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.936)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1268      84
True 1      58     289

AUC: 0.936

=== All results and summaries saved successfully ===
Running Neural Net...
Using parameters: {'hidden_layer_sizes': '128,64', 'alpha': 0.0001, 'learning_rate_

c:\Users\janst\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:199: UserWarning: [19:18:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [5]:
# ZOO attack on each model

import xgboost as xgb
from art.estimators.classification import SklearnClassifier, XGBoostClassifier
from art.attacks.evasion import ZooAttack

zoo_kwargs = dict(
    max_iter=10,
    binary_search_steps=5,
    initial_const=1e-3,
    learning_rate=1e-2,
    nb_parallel=1,
    batch_size=1,   # ZOO requires batch_size=1 for tabular
    targeted=False,
)

results = {}

# Limit samples per model for speed
MAX_SAMPLES = 200


def make_art_classifier(model, X, y, clip_values=(0, 1)):
    """
    Wraps the model in the correct ART classifier.
    Uses XGBoostClassifier for XGBoost models, SklearnClassifier otherwise.
    """
    # XGBoost branch
    if isinstance(model, xgb.XGBClassifier):
        print("Wrapping model with ART XGBoostClassifier")
        nb_classes = int(len(np.unique(y)))
        nb_features = int(X.shape[1])
        return XGBoostClassifier(
            model=model,
            nb_features=nb_features,   # IMPORTANT: use nb_features, not features
            nb_classes=nb_classes,
            clip_values=clip_values,
        )

    # Default: sklearn-compatible models
    print("Wrapping model with ART SklearnClassifier")
    return SklearnClassifier(model=model, clip_values=clip_values)


for name, (model, X_test, y_test) in model_runs.items():
    print("\n=== ZOO attack on model:", name, "===")

    # Convert data to numpy
    if hasattr(X_test, "to_numpy"):
        X_all = X_test.to_numpy().astype(np.float32)
    else:
        X_all = np.asarray(X_test, dtype=np.float32)

    if hasattr(y_test, "to_numpy"):
        y_all = y_test.to_numpy()
    else:
        y_all = np.asarray(y_test)

    # Optional: limit number of samples
    n_total = len(X_all)
    n = min(MAX_SAMPLES, n_total)
    X = X_all[:n]
    y = y_all[:n]

    # Ensure labels are integer-encoded 1D
    if isinstance(y, pd.Series):
        y = y.to_numpy()
    y_int = y.astype(int).reshape(-1)

    n = X.shape[0]

    print(f"Using {n} samples for ZOO on {name}")

    # Build the appropriate ART classifier (XGBoost or Sklearn)
    art_classifier = make_art_classifier(model, X, y_int, clip_values=(0, 1))

    # Create the ZOO attack instance
    attack = ZooAttack(classifier=art_classifier, **zoo_kwargs)

    clean_correct = 0
    adv_correct = 0

    for i in range(n):
        xi = X[i : i + 1]
        yi = y_int[i : i + 1]

        # Predictions on clean input
        pred_clean = np.argmax(art_classifier.predict(xi), axis=1)

        # Generate adversarial example for this single sample
        x_adv = attack.generate(x=xi, y=yi)

        # Predictions on adversarial input
        pred_adv = np.argmax(art_classifier.predict(x_adv), axis=1)

        # All three are 1-element arrays; compare scalars
        clean_correct += int(pred_clean[0] == yi[0])
        adv_correct   += int(pred_adv[0]   == yi[0])

    clean_acc = clean_correct / n
    adv_acc = adv_correct / n

    print(f"{name} clean acc: {clean_acc:.4f}")
    print(f"{name} adv   acc: {adv_acc:.4f}")

    results[name] = dict(clean_acc=float(clean_acc), adv_acc=float(adv_acc))

results_df = pd.DataFrame(results).T
results_df



=== ZOO attack on model: LogisticRegression ===
Using 200 samples for ZOO on LogisticRegression
Wrapping model with ART SklearnClassifier


ZOO: 100%|██████████| 1/1 [00:00<00:00, 111.02it/s]


LogisticRegression clean acc: 0.9100
LogisticRegression adv   acc: 0.1650

=== ZOO attack on model: NeuralNet ===
Using 200 samples for ZOO on NeuralNet
Wrapping model with ART SklearnClassifier


ZOO: 100%|██████████| 1/1 [00:00<00:00, 99.91it/s]


NeuralNet clean acc: 0.9600
NeuralNet adv   acc: 0.2650

=== ZOO attack on model: RandomForest ===
Using 200 samples for ZOO on RandomForest
Wrapping model with ART SklearnClassifier


ZOO:   0%|          | 0/1 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ================================================================
#   SKLEARN METRICS SUMMARY FOR CLEAN + ZOO PERFORMANCE
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np
import pandas as pd

zoo_summary_rows = []

print("\n\n===================== ZOO METRICS SUMMARY =====================\n")

for name, (model, X_test, y_test) in model_runs.items():

    print("\n\n################################################################")
    print("MODEL:", name)
    print("################################################################")

    # Convert labels
    if hasattr(y_test, "to_numpy"):
        y_true = y_test.to_numpy()
    else:
        y_true = np.asarray(y_test)

    # Clean predictions from the trained model
    y_pred = model.predict(X_test)

    # Basic clean metrics
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print("\nCLEAN PERFORMANCE")
    print("----------------------------")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # Row for summary table
    row = {
        "Model": name,
        "Clean Accuracy": acc,
        "Clean F1": f1
    }

    # ============================
    #          ZOO RESULTS
    # ============================
    if "results" in globals() and name in results:
        zoo_clean = results[name]["clean_acc"]
        zoo_adv   = results[name]["adv_acc"]
        zoo_drop  = zoo_clean - zoo_adv

        print("\nZOO ADVERSARIAL RESULTS")
        print("----------------------------")
        print("Clean Accuracy (ZOO dict):", zoo_clean)
        print("Adv Accuracy   (ZOO):     ", zoo_adv)
        print("Accuracy Drop:", zoo_drop)

        row["ZOO Adv Accuracy"] = zoo_adv
        row["ZOO Drop"] = zoo_drop
    else:
        print("\nNo ZOO results recorded for this model in results.")

    zoo_summary_rows.append(row)

# Create dataframe summary
zoo_metrics_summary_df = pd.DataFrame(zoo_summary_rows)
print("\n\n===================== ZOO SUMMARY DATAFRAME =====================\n")
display(zoo_metrics_summary_df)

zoo_metrics_summary_df




===================== ZOO METRICS SUMMARY =====================



################################################################
MODEL: LogisticRegression
################################################################

CLEAN PERFORMANCE
----------------------------
Accuracy: 0.9164214243672749
Precision: 0.9191983360683207
Recall: 0.9164214243672749
F1 Score: 0.9175247607419302

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699

Confusion Matrix:
[[1268   84]
 [  58  289]]

No ZOO results recorded for this model in results.


################################################################
MODEL: NeuralNet
################################################################

CLEAN PERFORMA

,Model,Clean Accuracy,Clean F1
0,LogisticRegression,0.916421,0.917525
1,NeuralNet,0.945851,0.945674
2,RandomForest,0.945262,0.943670
3,SVM,0.929880,0.925855
4,XGBoost,0.943496,0.942108


,Model,Clean Accuracy,Clean F1
0,LogisticRegression,0.916421,0.917525
1,NeuralNet,0.945851,0.945674
2,RandomForest,0.945262,0.943670
3,SVM,0.929880,0.925855
4,XGBoost,0.943496,0.942108


In [ ]:
# ==========================================
# MANUAL ADVERSARIAL DETECTOR FOR ZOO (SKLEARN ONLY)
# ==========================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import ZooAttack
from art.utils import check_and_transform_label_format
from xgboost import XGBClassifier  # so we can skip these


def _predict_labels_art(art_clf, x):
    """
    Safe wrapper: if ART returns probabilities, take argmax;
    if it returns labels, use them directly.
    """
    preds = np.asarray(art_clf.predict(x))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)


def run_zoo_manual_detector(model_runs, max_samples=1000):
    """
    For each sklearn model in model_runs:
      - run ZOO on up to max_samples points
      - build a detection dataset (clean=0, adv=1)
      - train LogisticRegression as a detector
      - report detector TPR/FPR + adv accuracy drop

    XGBoost models are skipped to avoid ART/ZOO incompatibilities.
    """
    rows = []

    for name, (model, X_test, y_test) in model_runs.items():
        # Skip XGBoost models (ART + ZOO is unstable for these in your setup)
        if isinstance(model, XGBClassifier):
            print(f"\n=== {name}: XGBoost model skipped for ZOO detector ===")
            continue

        print(f"\n=== {name}: ZOO + MANUAL DETECTOR ===")

        # Convert data
        X = X_test.to_numpy().astype(np.float32) if hasattr(X_test, "to_numpy") else np.asarray(X_test, dtype=np.float32)
        y_raw = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.asarray(y_test)

        # Map labels to 0..K-1 for ART
        classes, y_int = np.unique(y_raw, return_inverse=True)

        n = min(max_samples, len(X))
        if n == 0:
            print(f"{name}: no samples, skipping")
            continue

        X = X[:n]
        y = y_int[:n]

        clip_values = (float(X.min()), float(X.max()))
        art_clf = SklearnClassifier(model=model, clip_values=clip_values)

        # ZOO needs one-hot labels
        y_one_hot = check_and_transform_label_format(y, nb_classes=len(classes))

        # Configure ZOO (keep it simple to avoid version issues)
        zoo = ZooAttack(classifier=art_clf, max_iter=10, binary_search_steps=1, nb_parallel=1, batch_size=1)

        clean_correct = 0
        adv_correct = 0
        X_clean_list, X_adv_list = [], []

        for i in range(n):
            xi = X[i:i+1]
            yi_int = np.array([y[i]])
            yi_oh = y_one_hot[i:i+1]

            # clean prediction
            pred_clean = _predict_labels_art(art_clf, xi)

            # generate adversarial example for this point
            x_adv = zoo.generate(x=xi, y=yi_oh)

            # adv prediction
            pred_adv = _predict_labels_art(art_clf, x_adv)

            clean_correct += int(pred_clean[0] == yi_int[0])
            adv_correct += int(pred_adv[0] == yi_int[0])

            X_clean_list.append(xi)
            X_adv_list.append(x_adv)

        clean_acc = clean_correct / n
        adv_acc = adv_correct / n
        acc_drop = clean_acc - adv_acc

        # Build detection dataset: 0 = clean, 1 = adv
        X_clean_det = np.vstack(X_clean_list)
        X_adv_det = np.vstack(X_adv_list)
        X_det = np.vstack([X_clean_det, X_adv_det])
        y_det = np.concatenate([
            np.zeros(len(X_clean_det), dtype=int),
            np.ones(len(X_adv_det), dtype=int),
        ])

        # Manual detector: Logistic Regression
        detector = LogisticRegression(max_iter=1000)
        detector.fit(X_det, y_det)

        # Evaluate detector
        clean_preds = detector.predict(X_clean_det)
        adv_preds = detector.predict(X_adv_det)

        det_fpr = float((clean_preds == 1).mean())  # clean flagged as adv
        det_tpr = float((adv_preds == 1).mean())    # adv correctly flagged

        print(f"{name}: clean_acc={clean_acc:.4f}, adv_acc={adv_acc:.4f}, drop={acc_drop:.4f}")
        print(f"{name}: Detector TPR={det_tpr:.4f}, FPR={det_fpr:.4f}")

        rows.append({
            "Model": name,
            "DetectorTPR": det_tpr,
            "DetectorFPR": det_fpr,
            "CleanAccSubset": clean_acc,
            "AdvAccSubset": adv_acc,
            "AccDropSubset": acc_drop,
            "NumAttacked": n,
        })

    return pd.DataFrame(rows)


zoo_manual_detector_df = run_zoo_manual_detector(model_runs, max_samples=50)
display(zoo_manual_detector_df)
zoo_manual_detector_df



=== LogisticRegression: ZOO + MANUAL DETECTOR ===


ZOO: 100%|██████████| 1/1 [00:00<00:00, 398.21it/s]


LogisticRegression: clean_acc=0.9400, adv_acc=0.6600, drop=0.2800
LogisticRegression: Detector TPR=0.3200, FPR=0.3200

=== NeuralNet: ZOO + MANUAL DETECTOR ===


ZOO: 100%|██████████| 1/1 [00:00<00:00, 499.62it/s]
C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


NeuralNet: clean_acc=0.9800, adv_acc=0.7000, drop=0.2800
NeuralNet: Detector TPR=0.5000, FPR=0.4800

=== RandomForest: ZOO + MANUAL DETECTOR ===


...

RandomForest: clean_acc=0.9800, adv_acc=0.9800, drop=0.0000
RandomForest: Detector TPR=0.0000, FPR=0.0000

=== SVM: ZOO + MANUAL DETECTOR ===


ZOO: 100%|██████████| 1/1 [00:00<00:00, 493.45it/s]

SVM: clean_acc=0.9400, adv_acc=0.8400, drop=0.1000
SVM: Detector TPR=0.0000, FPR=0.0000

=== XGBoost: XGBoost model skipped for ZOO detector ===


,Model,DetectorTPR,DetectorFPR,CleanAccSubset,AdvAccSubset,AccDropSubset,NumAttacked
0,LogisticRegression,0.32,0.32,0.94,0.66,0.28,50
1,NeuralNet,0.50,0.48,0.98,0.70,0.28,50
2,RandomForest,0.00,0.00,0.98,0.98,0.00,50
3,SVM,0.00,0.00,0.94,0.84,0.10,50


,Model,DetectorTPR,DetectorFPR,CleanAccSubset,AdvAccSubset,AccDropSubset,NumAttacked
0,LogisticRegression,0.32,0.32,0.94,0.66,0.28,50
1,NeuralNet,0.50,0.48,0.98,0.70,0.28,50
2,RandomForest,0.00,0.00,0.98,0.98,0.00,50
3,SVM,0.00,0.00,0.94,0.84,0.10,50
